# Baseline: CQT + Global Average + Cosine Similarity

Classical MIR baseline for music retrieval. Uses Constant-Q Transform (CQT) with temporal averaging and cosine similarity.

**Limitations:** Fails under tempo changes, key shifts, and short queries.

In [4]:
import librosa
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

# Resolve project root (works when run from project root or notebooks/)
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

def extract_cqt_embedding(path):
    """Extract CQT-based embedding with global average pooling."""
    y, sr = librosa.load(path, sr=22050)
    cqt = librosa.cqt(y, sr=sr)
    cqt = np.abs(cqt)
    embedding = np.mean(cqt, axis=1)
    return embedding

In [6]:
# Build database of CQT embeddings
database = {}
for f in sorted(PROCESSED_DIR.glob("*.wav")):
    emb = extract_cqt_embedding(str(f))
    database[f.name] = emb

print(f"Indexed {len(database)} pieces")

Indexed 100 pieces


In [5]:
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("Files found:", list(PROCESSED_DIR.glob("*.wav"))[:5])

PROJECT_ROOT: /scratch2/ychen17/AI_Music
PROCESSED_DIR: /scratch2/ychen17/AI_Music/data/processed
Files found: [PosixPath('/scratch2/ychen17/AI_Music/data/processed/piece_001.wav'), PosixPath('/scratch2/ychen17/AI_Music/data/processed/piece_002.wav'), PosixPath('/scratch2/ychen17/AI_Music/data/processed/piece_003.wav'), PosixPath('/scratch2/ychen17/AI_Music/data/processed/piece_004.wav'), PosixPath('/scratch2/ychen17/AI_Music/data/processed/piece_005.wav')]


In [ ]:
# Query: use a snippet as query
query_path = str(PROCESSED_DIR / "piece_001.wav")  # testing with piece_001.wav
query_emb = extract_cqt_embedding(query_path)

scores = []
for name, emb in database.items():
    sim = cosine_similarity(
        query_emb.reshape(1, -1),
        emb.reshape(1, -1)
    )[0][0]
    scores.append((name, sim))

scores.sort(key=lambda x: x[1], reverse=True)
print("Top 5 matches:")
for name, sim in scores[:5]:
    print(f"  {name}: {sim:.4f}")

Top 5 matches:
  piece_001.wav: 1.0000
  piece_098.wav: 0.9895
  piece_072.wav: 0.9857
  piece_066.wav: 0.9853
  piece_064.wav: 0.9850
